# dr_evt 06: the `dr_evt_market` package, demonstrated

Notebooks 02 to 04 drove dr_evt by hand: raw `Simulation` calls in process, raw
`ClientMessage`s over gRPC, and a start-time rule written inline. The `dr_evt_market`
package under `python/` is the same material made reusable: one contract
(`PlatformSession`), two adapters that implement it (`InProcessPlatform`,
`GrpcPlatform`), a server launcher (`ServerProcess`) and an error taxonomy. This
notebook runs each of those on the contended ten-job
stream the package tests use, and checks every claim against dr_evt's own `simulator`
binary.

Sections:

1. one platform in process: submit, advance, snapshot, timings, finish
2. the same platform over gRPC, on a server the package starts and stops
3. the error taxonomy: what the adapters refuse before dr_evt sees it
4. parity: in process and over the wire give the same schedule as the CLI

In [1]:
from pathlib import Path
import os, sys, tempfile, time
import pandas as pd

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
sys.path[:0] = [str(INSTALL / "lib" / "python"), str(DR_EVT / "python")]   # dr_evt extension, then the package

import dr_evt_market as m
from dr_evt_market.tests.fixtures import CONTENDED_JOBS, EXPECTED_BEGIN_TIMES, cli_schedule
print("dr_evt_market", m.__version__, "| exports:", ", ".join(n for n in m.__all__))

dr_evt_market 0.1.0 | exports: __version__, ClockViolation, ConfigurationError, InfrastructureFailure, GrpcPlatform, InProcessPlatform, JobTiming, PlatformReport, PlatformSession, PlatformSnapshot, ServerProcess, SessionClient, StructuralRejection, SubmitRequest, validate


## 1. One platform in process

`InProcessPlatform(name, total_nodes, work_dir)` owns one `dr_evt.Simulation`. The
work directory receives the header-only input CSV dr_evt insists on parsing, and later
the two output files. The stream below is the package's fixture: ten jobs, one every
10 s, sized so that they contend for a 100-node platform (the fourth job needs 40 nodes
and has to wait for the first three to finish).

In [2]:
WORK = Path(tempfile.mkdtemp(dir=OUT, prefix="market_"))
jobs = pd.DataFrame([{"key": j.key, "submit_s": j.submit_s, "nodes": j.num_nodes, "limit_s": j.limit_s} for j in CONTENDED_JOBS])
jobs.T

,0,1,2,3,4,5,6,7,8,9
key,job-0,job-1,job-2,job-3,job-4,job-5,job-6,job-7,job-8,job-9
submit_s,0,10,20,30,40,50,60,70,80,90
nodes,20,30,15,40,25,10,35,60,20,45
limit_s,200,150,300,100,250,80,180,220,90,160


In [3]:
alpha = m.InProcessPlatform("alpha", 100, WORK / "alpha")
handles = alpha.submit(CONTENDED_JOBS[:3])       # submit only queues ...
print("after submit:  now =", alpha.now(), "| waiting =", alpha.snapshot().waiting_jobs, "| free =", alpha.snapshot().free_nodes)
alpha.advance_to(0)                              # ... advancing evaluates the queue
s = alpha.snapshot()
print("after advance: now =", alpha.now(), "| waiting =", s.waiting_jobs, "| free =", s.free_nodes, "| in use =", s.in_use_nodes)
print("handles are dr_evt's job indices:", handles)

after submit:  now = 0 | waiting = 1 | free = 100
after advance: now = 0 | waiting = 0 | free = 80 | in use = 20
handles are dr_evt's job indices: [0, 1, 2]


`snapshot()` is what a routing decision reads: free nodes above all, plus the queue
length and the instantaneous utilization. The market allocates only free nodes, so this
is the whole supply side of one platform at one time.

In [4]:
alpha.submit(CONTENDED_JOBS[3:])
alpha.advance_to(30)
s = alpha.snapshot()
pd.Series({"time_s": s.time_s, "total_nodes": s.total_nodes, "free_nodes": s.free_nodes, "in_use_nodes": s.in_use_nodes,
           "waiting_jobs": s.waiting_jobs, "current_utilization": s.current_utilization})

time_s                  30.00
total_nodes            100.00
free_nodes              35.00
in_use_nodes            65.00
waiting_jobs             1.00
current_utilization      0.65
dtype: float64

At 30 s the fourth job (40 nodes) is waiting: three jobs hold 65 nodes and it does not
fit, so dr_evt holds it at the head of its queue. The market never creates this
situation on purpose, since it submits only what fits now and leaves the queue empty;
the fixture stream submits everything regardless of capacity so that the queue is
visible here.

`timings(handles)` is PR 1's per-job accessor: one record per handle, with the client's
own key attached. A job that has not started yet reports `scheduled=False` and -1 times.
`finish()` drains the platform, writes the simulated trace and the resource history, and
returns everything as a `PlatformReport`.

In [5]:
all_handles = list(range(len(CONTENDED_JOBS)))
pd.DataFrame([t.__dict__ for t in alpha.timings(all_handles)])[["handle", "key", "submit_s", "begin_s", "end_s", "scheduled"]]

,handle,key,submit_s,begin_s,end_s,scheduled
0,0,job-0,0.0,0.0,200.0,True
1,1,job-1,10.0,10.0,160.0,True
2,2,job-2,20.0,20.0,320.0,True
3,3,job-3,30.0,-1.0,-1.0,False
4,4,job-4,40.0,-1.0,-1.0,False
5,5,job-5,50.0,-1.0,-1.0,False
6,6,job-6,60.0,-1.0,-1.0,False
7,7,job-7,70.0,-1.0,-1.0,False
8,8,job-8,80.0,-1.0,-1.0,False
9,9,job-9,90.0,-1.0,-1.0,False


In [6]:
report = alpha.finish()
timings = pd.DataFrame([t.__dict__ for t in report.timings])
print("statistics:", {k: round(v, 3) for k, v in report.statistics.items() if k in ("jobs_completed", "makespan", "avg_wait_time", "utilization")})
print("files:", Path(report.simulated_trace_path).name, Path(report.resource_trace_path).name)
print("begin times:", timings.begin_s.astype(int).tolist(), "| expected:", list(EXPECTED_BEGIN_TIMES))
assert timings.begin_s.astype(int).tolist() == list(EXPECTED_BEGIN_TIMES)

statistics: {'jobs_completed': 10.0, 'utilization': 0.665, 'avg_wait_time': 151.0, 'makespan': 790.0}
files: alpha.simulated.csv alpha.resource.csv
begin times: [0, 10, 20, 160, 160, 50, 260, 410, 260, 630] | expected: [0, 10, 20, 160, 160, 50, 260, 410, 260, 630]


## 2. The same platform over gRPC

`ServerProcess` starts `dr_evt_server` in a working directory on a free port, waits for
the channel, and stops it on exit; its output goes to `server.log` in that directory.
`GrpcPlatform` is the in-process adapter call for call, but every method is one or more
messages on the session stream. The generated stubs are produced from
`src/proto/dr_evt_service.proto` at construction (nothing generated is checked in).

In [7]:
SERVER_DIR = WORK / "server"
with m.ServerProcess(None, SERVER_DIR) as server:                  # None: find the binary under the install prefix
    print("server at", server.address)
    beta = m.GrpcPlatform("beta", 100, server.address, SERVER_DIR, session_name="demo")
    for job in CONTENDED_JOBS:
        beta.advance_to(job.submit_s); beta.submit([job]); beta.advance_to(job.submit_s)
    s = beta.snapshot()
    print("at", s.time_s, "s: free", s.free_nodes, "| waiting", s.waiting_jobs, "| utilization", s.current_utilization)
    remote = beta.finish()
print("server stopped; files:", sorted(p.name for p in SERVER_DIR.iterdir()))
remote_timings = pd.DataFrame([t.__dict__ for t in remote.timings])
assert remote_timings.begin_s.tolist() == timings.begin_s.tolist()
print("gRPC begin times equal in-process begin times:", remote_timings.begin_s.astype(int).tolist())

server at 127.0.0.1:60217
at 90 s: free 25 | waiting 6 | utilization 0.75
server stopped; files: ['demo-1789596201655639-0-976994945.resource.csv', 'demo-1789596201655639-0-976994945.simulated.csv', 'demo-1789596201655639-0-976994945.statistics.json', 'demo.header.csv', 'server.log']
gRPC begin times equal in-process begin times: [0, 10, 20, 160, 160, 50, 260, 410, 260, 630]


## 3. The error taxonomy

dr_evt is permissive: it accepts a job larger than the platform and drops it silently, it
does not check that submit times are non-decreasing, and it truncates fractional seconds.
The adapters refuse all of that before dr_evt sees it, with four exception types the
market can catch by meaning rather than by message.

In [8]:
delta = m.InProcessPlatform("delta", 100, WORK / "delta")
delta.submit([m.SubmitRequest("ok", 0, 10, 100)]); delta.advance_to(10)
attempts = {
    "larger than the platform": lambda: delta.submit([m.SubmitRequest("big", 10, 101, 100)]),
    "submitted in the past":    lambda: delta.submit([m.SubmitRequest("late", 5, 10, 100)]),
    "fractional submit time":   lambda: delta.submit([m.SubmitRequest("frac", 10.5, 10, 100)]),
    "queue id not numeric":     lambda: delta.submit([m.SubmitRequest("q", 10, 10, 100, q_id="pbatch")]),
    "advance backwards":        lambda: delta.advance_to(5),
    "unknown timing handle":    lambda: delta.timings([99]),
}
for what, attempt in attempts.items():
    try:
        attempt(); print(f"{what:28s} -> accepted (unexpected)")
    except Exception as e:
        print(f"{what:28s} -> {type(e).__name__}: {e}")
print("queue untouched: waiting =", delta.snapshot().waiting_jobs)

larger than the platform     -> StructuralRejection: job 'big' requests 101 nodes, but platform 'delta' has 100
submitted in the past        -> ClockViolation: job 'late' submits at 5, before platform time 10
fractional submit time       -> ClockViolation: submit_s must be an integer number of seconds
queue id not numeric         -> ConfigurationError: q_id must be a digit string in 1..10
advance backwards            -> ClockViolation: cannot advance platform 'delta' from 10 back to 5
unknown timing handle        -> KeyError: "platform 'delta' has no job handle 99"
queue untouched: waiting = 0


## 4. Parity with the CLI

The package's own oracle is dr_evt's `simulator` binary on the same ten jobs. The test
suite asserts this too (`tests/run_market_tests.sh`, 23 tests); here it is by hand for
both transports.

In [9]:
cli = pd.DataFrame(cli_schedule(WORK / "cli", CONTENDED_JOBS))          # runs install/bin/simulator on the same ten jobs
compare = pd.DataFrame({"in_process": timings.begin_s.tolist(), "grpc": remote_timings.begin_s.tolist(),
                        "cli": cli.begin_time.astype(float).tolist(), "expected": [float(v) for v in EXPECTED_BEGIN_TIMES]})
print(compare.T.to_string())
assert (compare.nunique(axis=1) == 1).all(), "transports disagree"
print("all columns identical for every job")

              0     1     2      3      4     5      6      7      8      9
in_process  0.0  10.0  20.0  160.0  160.0  50.0  260.0  410.0  260.0  630.0
grpc        0.0  10.0  20.0  160.0  160.0  50.0  260.0  410.0  260.0  630.0
cli         0.0  10.0  20.0  160.0  160.0  50.0  260.0  410.0  260.0  630.0
expected    0.0  10.0  20.0  160.0  160.0  50.0  260.0  410.0  260.0  630.0
all columns identical for every job


## What comes next

The package stops at the platform boundary on purpose: it can run any platform, submit
to it and read the truth back, but it does not yet decide anything. The next pull
request adds the client that is the auction: a jobs file with one bid per platform,
the observation a mechanism sees (queued jobs, bids, free nodes per platform), an
abstract `Mechanism` with VCG as its first subclass, and the loop that clears one
window at a time, submits the winners, and checks from the timing records shown above
that every routed job began at its window time. Notebook 07 will run that pipeline end
to end on the platforms shown here.